In [1]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/faster_rcnn
%cp VOC2007.zip /content
%cp VOC2012.zip /content
%cd /content

Mounted at /content/drive
/content/drive/MyDrive/faster_rcnn
/content


In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
!nvidia-smi

Fri Jul 17 09:35:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |    7415MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from src.backbone import Backbone
from src.rpn import RPN_Head, RPN_Loss

backbone = Backbone()
rpn_head = RPN_Head(in_channels=1024, mid_channels=512)

In [6]:
loss_criterion = RPN_Loss()

params = list(backbone.parameters()) + list(rpn_head.parameters())
optimizer = torch.optim.SGD(
    params, 
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005)

In [7]:
backbone.to(device)
rpn_head.to(device)

RPN_Head(
  (conv1): Conv2d(1024, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv_cls): Conv2d(512, 18, kernel_size=(1, 1), stride=(1, 1))
  (conv_reg): Conv2d(512, 36, kernel_size=(1, 1), stride=(1, 1))
)

In [11]:
from pathlib import Path

start_epoch = 0

checkpoint_dir = Path("/content/drive/MyDrive/faster_rcnn/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

existing_checkpoints = sorted(checkpoint_dir.glob("step1_epoch_*.pt"),
                              key = lambda p : p.stem.split("_epoch_")[1])

if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")

    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    backbone.load_state_dict(checkpoint['backbone_state_dict'])
    rpn_head.load_state_dict(checkpoint['rpn_head_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1

else:
    print("No existing checkpoints found. Starting training from scratch.")


Loading checkpoint: /content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_4.pt


In [9]:
from src.backbone import backbone_transform
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

transform = backbone_transform

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

combined_train_img_paths = voc2007_img_paths_train + voc2012_img_paths_train

train_dataloader = create_voc_dataloader(img_paths_list=combined_train_img_paths, transform=transform, batch_size=2, shuffle=True)

In [ ]:
import torch
import time
from tqdm import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed(42)

num_epochs = 8
loss_lambda = 10

backbone.train()
rpn_head.train()

for epoch in range(start_epoch, num_epochs):
    start_time = time.time()

    epoch_cls_loss = 0.0
    epoch_reg_loss = 0.0
    num_batches = 0

    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

    for batch_imgs, batch_gt_boxes, batch_gt_labels, batch_img_sizes_before_pad in progress_bar:
        batch_imgs = batch_imgs.to(device)
        batch_gt_boxes = [boxes.to(device) for boxes in batch_gt_boxes]         # Not needed as the functions take care, but for safety
        batch_gt_labels = [labels.to(device) for labels in batch_gt_labels]

        batch_feature_maps = backbone(batch_imgs)

        batch_cls_logits, batch_rpn_box_deltas, batch_anchors = rpn_head(batch_feature_maps, batch_img_height=batch_imgs.shape[2], batch_img_width=batch_imgs.shape[3])

        cls_loss, reg_loss = loss_criterion(batch_cls_logits,
                                            batch_rpn_box_deltas, 
                                            batch_anchors, 
                                            batch_gt_boxes, 
                                            batch_img_sizes_before_pad)
        
        loss = cls_loss + loss_lambda * reg_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_cls_loss += cls_loss.item()
        epoch_reg_loss += reg_loss.item()
        num_batches += 1

        progress_bar.set_postfix({
            "cls_loss": f"{cls_loss.item():.4f}",
            "reg_loss": f"{reg_loss.item():.4f}",
            "total_loss": f"{loss.item():.4f}"
        })

    avg_cls_loss = epoch_cls_loss / num_batches
    avg_reg_loss = epoch_reg_loss / num_batches

    print(f"Epoch {epoch+1}/{num_epochs} completed in {time.time() - start_time:.2f}s")
    print(f"Average Classification Loss: {avg_cls_loss:.4f}")
    print(f"Average Regression Loss: {avg_reg_loss:.4f}")

    checkpoint_path = checkpoint_dir / f"step1_epoch_{epoch+1}.pt"
    torch.save({
        'epoch': epoch,
        'backbone_state_dict': backbone.state_dict(),
        'rpn_head_state_dict': rpn_head.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, checkpoint_path)

    print(f"Checkpoint saved at: {checkpoint_path}")

Epoch 5/8: 100%|██████████| 8276/8276 [39:32<00:00,  3.49batch/s, cls_loss=0.3337, reg_loss=0.0090, total_loss=0.4240] 


Epoch 5/8 completed in 2372.22s
Average Classification Loss: 0.0773
Average Regression Loss: 0.0684
Checkpoint saved at: /content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_5.pt


Epoch 6/8:  76%|███████▌  | 6272/8276 [29:48<09:13,  3.62batch/s, cls_loss=0.1564, reg_loss=0.0421, total_loss=0.5778] 